# Film Category Prediction (horror / comedy)

1. Audit and cleaning
2. Dataset split
3. Text representation (TF-IDF)
4. Comparison baselines
5. Hyperparameter search with KerasTuner
6. Hyperparameter search with Optuna
7. Verdict on the test set
8. Saving

In [3]:
import json
import platform
import warnings
from datetime import datetime
from pathlib import Path

import joblib
import keras_tuner as kt
import numpy as np
import optuna
import pandas as pd
import sklearn
import spacy
import tensorflow as tf
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from tensorboard.plugins.hparams import api as hparams_api
import keras

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = Path("/home/projects/predict-film-category/data/textes_films_horreur_comedie.csv")
MODEL_DIR = Path("model")
TB_DIR = Path("my_log_dir") / datetime.now().strftime("%Y%m%d-%H%M%S")  # TensorBoard logs, one folder per session
SEED = 42
SEEDS = [0, 1, 2, 3, 4]
LABELS = {1: "horror", 0: "comedy"}

try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

I0000 00:00:1793197267.054398  197730 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## 1. Audit and cleaning

In [4]:
# Columns of the source file: "resume" (film summary) and "cible" (target: 1 = horror, 0 = comedy)
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print(df["cible"].map(LABELS).value_counts().to_string())
print()
print(df["resume"].str.len().describe().round(0).to_string())

Shape: (2400, 3)
cible
comedy    1200
horror    1200

count    2400.0
mean      558.0
std       349.0
min       201.0
25%       335.0
50%       474.0
75%       679.0
max      7104.0


In [5]:
n_before = len(df)

df = df.dropna(subset=["resume"])
df["resume"] = df["resume"].str.strip()
df = df[df["resume"] != ""]

n_duplicates = int(df["resume"].duplicated().sum())
n_conflicting = int((df.groupby("resume")["cible"].nunique() > 1).sum())
df = df.drop_duplicates(subset="resume").reset_index(drop=True)

print(f"{n_before} rows | {n_duplicates} duplicates removed | {n_conflicting} summary(ies) with conflicting targets")
print(f"{len(df)} texts kept")
print(df["cible"].map(LABELS).value_counts().to_string())

2400 rows | 54 duplicates removed | 0 summary(ies) with conflicting targets
2346 texts kept
cible
comedy    1195
horror    1151


In [6]:
def clean_doc(doc):
    """Lowercase lemmas, alphabetic tokens only, stop words removed."""
    return " ".join(t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop)


# Stateless transformation: no leakage between training and test
df["clean"] = [clean_doc(d) for d in nlp.pipe(df["resume"], batch_size=128)]
df = df[df["clean"].str.len() > 0].reset_index(drop=True)

print(df["clean"].iloc[0][:200])

roberto new job teacher nursery school child meet gianluigi mute roberto spirited bring tv class donkey take child unscheduled field trip father plant date isabella mother student soon pregnant moody 


## 2. Dataset split

In [7]:
# 60% training / 20% test / 20% production (held out)
train, rest = train_test_split(df, test_size=0.40, stratify=df["cible"], random_state=SEED)
test, prod = train_test_split(rest, test_size=0.50, stratify=rest["cible"], random_state=SEED)

for name, part in (("training", train), ("test", test), ("production", prod)):
    print(f"{name:<12} {len(part):>5} texts | horror share: {part['cible'].mean():.3f}")

y_train, y_test, y_prod = (p["cible"].to_numpy() for p in (train, test, prod))

training      1407 texts | horror share: 0.490
test           469 texts | horror share: 0.490
production     470 texts | horror share: 0.491


## 3. Text representation (TF-IDF)

In [8]:
vec = TfidfVectorizer()
X_train = vec.fit_transform(train["clean"])  # vocabulary learned on the training set only
X_test = vec.transform(test["clean"])
X_prod = vec.transform(prod["clean"])

# Dense version for the neural networks
X_train_d, X_test_d, X_prod_d = (m.toarray().astype("float32") for m in (X_train, X_test, X_prod))

print("Vocabulary:", len(vec.vocabulary_), "words")

Vocabulary: 11807 words


## 4. Comparison baselines

In [9]:
def noise_margin(p, n):
    return 1.96 * np.sqrt(p * (1 - p) / n)


def validation_indices(seed):
    """Split the training set in two: the test set stays untouched."""
    return train_test_split(np.arange(len(y_train)), test_size=0.20, stratify=y_train, random_state=seed)


def train_network(model, X, y, run_name=None, histogram_freq=0):
    """Train with early stopping; if run_name is given, also log the run for TensorBoard."""
    callbacks = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)]
    if run_name:
        callbacks.append(keras.callbacks.TensorBoard(log_dir=str(TB_DIR / run_name), histogram_freq=histogram_freq))
    model.fit(X, y, epochs=40, batch_size=32, validation_split=0.10, callbacks=callbacks, verbose=0)
    return model


def predict_network(model, X):
    return (model.predict(X, verbose=0).ravel() > 0.5).astype(int)


def baseline_network():
    model = keras.Sequential([
        keras.Input(shape=(X_train_d.shape[1],)),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return model


def scores_sklearn(factory):
    scores = []
    for seed in SEEDS:
        i_fit, i_val = validation_indices(seed)
        model = factory().fit(X_train[i_fit], y_train[i_fit])
        scores.append(f1_score(y_train[i_val], model.predict(X_train[i_val])))
    return scores


def scores_network(factory, tag=None):
    scores = []
    for seed in SEEDS:
        keras.utils.set_random_seed(seed)
        i_fit, i_val = validation_indices(seed)
        run_name = f"{tag}/seed_{seed}" if tag else None
        model = train_network(factory(), X_train_d[i_fit], y_train[i_fit], run_name=run_name)
        scores.append(f1_score(y_train[i_val], predict_network(model, X_train_d[i_val])))
    return scores


def summary_table(results):
    n_val = len(validation_indices(0)[1])
    return pd.DataFrame([
        {"model": name, "mean F1": np.mean(s), "std": np.std(s), "noise margin": noise_margin(np.mean(s), n_val)}
        for name, s in results.items()
    ]).round(3)

In [10]:
results = {
    "Logistic regression": scores_sklearn(lambda: LogisticRegression(max_iter=2000)),
    "Naive Bayes": scores_sklearn(MultinomialNB),
    "Baseline network": scores_network(baseline_network, tag="baseline_network"),
}
summary_table(results)

,model,mean F1,std,noise margin
0,Logistic regression,0.842,0.018,0.043
1,Naive Bayes,0.859,0.023,0.041
2,Baseline network,0.840,0.019,0.043


## 5. Hyperparameter search with KerasTuner

In [11]:
# Search space shared by KerasTuner and Optuna: (min, max[, step])
SEARCH_SPACE = {
    "n_layers": (1, 3),
    "units": (32, 256, 32),
    "dropout": (0.0, 0.5, 0.1),
    "l2": (1e-6, 1e-2),
    "lr": (1e-4, 1e-2),
}
N_TRIALS = 20

i_fit, i_val = validation_indices(SEED)
X_fit, X_val = X_train_d[i_fit], X_train_d[i_val]
y_fit, y_val = y_train[i_fit], y_train[i_val]


def build_network(n_layers, units, dropout, l2, lr):
    model = keras.Sequential([keras.Input(shape=(X_train_d.shape[1],))])
    for _ in range(int(n_layers)):
        model.add(keras.layers.Dense(int(units), activation="relu", kernel_regularizer=keras.regularizers.l2(l2)))
        model.add(keras.layers.Dropout(dropout))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model


def log_hparams(run_name, params, f1):
    """Record the settings and the F1 of one trial (TensorBoard HParams tab)."""
    with tf.summary.create_file_writer(str(TB_DIR / run_name)).as_default():
        hparams_api.hparams(params)
        tf.summary.scalar("val_f1", f1, step=1)


def score_f1(model, run_name=None, params=None):
    """Same protocol for both tools: train on X_fit, measure F1 on X_val."""
    keras.utils.set_random_seed(SEED)
    train_network(model, X_fit, y_fit, run_name=run_name)
    f1 = f1_score(y_val, predict_network(model, X_val))
    if run_name and params:
        log_hparams(run_name, params, f1)
    return f1


class F1Search(kt.HyperModel):
    def __init__(self):
        super().__init__()
        self.n_trials = 0

    def build(self, hp):
        s = SEARCH_SPACE
        return build_network(
            n_layers=hp.Int("n_layers", s["n_layers"][0], s["n_layers"][1]),
            units=hp.Int("units", s["units"][0], s["units"][1], step=s["units"][2]),
            dropout=hp.Float("dropout", s["dropout"][0], s["dropout"][1], step=s["dropout"][2]),
            l2=hp.Float("l2", s["l2"][0], s["l2"][1], sampling="log"),
            lr=hp.Float("lr", s["lr"][0], s["lr"][1], sampling="log"),
        )

    def fit(self, hp, model, **kwargs):
        self.n_trials += 1
        run_name = f"kerastuner/trial_{self.n_trials:02d}"
        return {"val_f1": score_f1(model, run_name=run_name, params=hp.values)}

In [12]:
tuner = kt.RandomSearch(
    F1Search(),
    objective=kt.Objective("val_f1", "max"),
    max_trials=N_TRIALS,
    seed=SEED,
    directory="kt_logs",
    project_name="film_category",
    overwrite=True,
)
tuner.search()

params_kt = tuner.get_best_hyperparameters(1)[0].values
score_kt = tuner.oracle.get_best_trials(1)[0].score
print(f"KerasTuner: F1 = {score_kt:.3f} | {params_kt}")

Trial 20 Complete [00h 00m 25s]
val_f1: 0.8405797101449275

Best val_f1 So Far: 0.8582089552238806
Total elapsed time: 00h 06m 42s
KerasTuner: F1 = 0.858 | {'n_layers': 2, 'units': 32, 'dropout': 0.2, 'l2': 1.2232117875737861e-05, 'lr': 0.005977728042983696}


## 6. Hyperparameter search with Optuna

In [13]:
def objective(trial):
    s = SEARCH_SPACE
    params = {
        "n_layers": trial.suggest_int("n_layers", s["n_layers"][0], s["n_layers"][1]),
        "units": trial.suggest_int("units", s["units"][0], s["units"][1], step=s["units"][2]),
        "dropout": trial.suggest_float("dropout", s["dropout"][0], s["dropout"][1], step=s["dropout"][2]),
        "l2": trial.suggest_float("l2", s["l2"][0], s["l2"][1], log=True),
        "lr": trial.suggest_float("lr", s["lr"][0], s["lr"][1], log=True),
    }
    return score_f1(build_network(**params), run_name=f"optuna/trial_{trial.number:02d}", params=params)


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler(seed=SEED))
study.optimize(objective, n_trials=N_TRIALS)

params_optuna = study.best_params
score_optuna = study.best_value
print(f"Optuna: F1 = {score_optuna:.3f} | {params_optuna}")

Optuna: F1 = 0.855 | {'n_layers': 3, 'units': 224, 'dropout': 0.0, 'l2': 2.7155819552829413e-05, 'lr': 0.0001705053926026929}


In [14]:
margin = noise_margin(max(score_kt, score_optuna), len(y_val))
gap = abs(score_kt - score_optuna)

print(f"KerasTuner {score_kt:.3f} | Optuna {score_optuna:.3f} | gap {gap:.3f} | noise margin ±{margin:.3f}")
print("Gap below the noise margin: the tools are equivalent." if gap < margin
      else "Gap above the noise margin: one tool is better.")

best_params = params_kt if score_kt >= score_optuna else params_optuna

KerasTuner 0.858 | Optuna 0.855 | gap 0.003 | noise margin ±0.041
Gap below the noise margin: the tools are equivalent.


## 7. Verdict on the test set

In [15]:
# Comparison over 5 seeds (same protocol as stage 4)
results["Tuned network"] = scores_network(lambda: build_network(**best_params), tag="tuned_network")
summary = summary_table(results)
summary

,model,mean F1,std,noise margin
0,Logistic regression,0.842,0.018,0.043
1,Naive Bayes,0.859,0.023,0.041
2,Baseline network,0.840,0.019,0.043
3,Tuned network,0.846,0.020,0.042


In [16]:
means = {name: np.mean(s) for name, s in results.items()}
best_simple = max(
    ("Logistic regression", "Naive Bayes"),
    key=lambda name: means[name],
)
n_val = len(validation_indices(0)[1])

for reference in (best_simple, "Baseline network"):
    gap = means["Tuned network"] - means[reference]
    margin = noise_margin(means["Tuned network"], n_val)
    verdict = "better" if gap > margin else "gap below the noise margin" if abs(gap) <= margin else "worse"
    print(f"Tuned network vs {reference}: gap {gap:+.3f} (margin ±{margin:.3f}) -> {verdict}")

Tuned network vs Naive Bayes: gap -0.012 (margin ±0.042) -> gap below the noise margin
Tuned network vs Baseline network: gap +0.006 (margin ±0.042) -> gap below the noise margin


In [17]:
# Final training on the whole training set, single measurement on the test set
keras.utils.set_random_seed(SEED)
final_model = train_network(build_network(**best_params), X_train_d, y_train, run_name="final_model", histogram_freq=1)

pred_test = predict_network(final_model, X_test_d)
f1_test = f1_score(y_test, pred_test)
margin_test = noise_margin(f1_test, len(y_test))
print(f"Test F1: {f1_test:.3f} ± {margin_test:.3f} (n = {len(y_test)})")

W0000 00:00:1793198264.922501  197730 cpu_allocator_impl.cc:82] Allocation of 90677760 exceeds 10% of free system memory.
W0000 00:00:1793198265.743289  197730 cpu_allocator_impl.cc:82] Allocation of 90677760 exceeds 10% of free system memory.
W0000 00:00:1793198266.259422  197730 cpu_allocator_impl.cc:82] Allocation of 90677760 exceeds 10% of free system memory.
W0000 00:00:1793198266.800262  197730 cpu_allocator_impl.cc:82] Allocation of 90677760 exceeds 10% of free system memory.


Test F1: 0.848 ± 0.032 (n = 469)


### TensorBoard dashboard

In [1]:
#%load_ext tensorboard
#%tensorboard --logdir {TB_DIR}

## 8. Saving

In [19]:
MODEL_DIR = Path("/home/projects/predict-film-category/model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

final_model.save(MODEL_DIR / "model.keras")
joblib.dump(vec, MODEL_DIR / "tfidf.joblib")

metrics = {
    "model_version": "1.0.0",
    "f1_test": round(float(f1_test), 4),
    "test_size": int(len(y_test)),
    "noise_margin": round(float(margin_test), 4),
    "best_params": best_params,
    "library_versions": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "tensorflow": tf.__version__,
        "spacy": spacy.__version__,
        "joblib": joblib.__version__,
    },
}
with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False, default=lambda o: o.item())

train[["resume", "cible"]].sample(50, random_state=SEED).to_csv(
    MODEL_DIR / "reference_sample.csv", index=False, encoding="utf-8"
)

print(sorted(p.name for p in MODEL_DIR.iterdir()))

['metrics.json', 'model.keras', 'model2.png', 'reference_sample.csv', 'tfidf.joblib']
